# Random Forest

Il Random Forest è un algoritmo di machine learning supervisionato basato su una collezionei alberi decisionali indipendenti, combinati per migliorare la accuratezza e la robustezza rispetto a singoli alberi.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Treining

In [2]:
def training(file_path, csv_name):

    # Legge il dataset
    df = pd.read_csv(file_path)

    # Definisco le colonne target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']
    
    # Vado a rimuovere le lesioni (righe) non valide
    df_validi = df.dropna(subset=original_target_list).copy()

    # Binarizzazione forzata dei target per robustezza e per evitare errori
    # Questa operazione garantisce che i target siano sempre 0 o 1, indipendentemente
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)  # Soglia clinica comune per KI67
    
    # Lista finale delle colonne target binarizzate che verranno usate per l'addestramento
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']
    
    """
     Preparo le feature (X) e i target (y) per il modello
    """
    # Definisco tutte le colonne da rimuovere per ottenere solo le feature radiomiche
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    # 'target' contiene le 3 colonne da usare
    target = df_validi[final_target_list]
    
    # 'groups' contiene l'ID del paziente per ogni lesione.
    # Mi serve per fare la cross-validation a gruppo
    groups = df_validi['Patient ID']

    # Riempie eventuali valori mancanti (NaN) rimasti nelle colonne delle feature
    # Evito errori durante il training
    features = features.fillna(features.mean())


    # Istanzio il classificatore RandomForest con parametri standard
    # random_state=42 garantisce che i risultati siano riproducibili
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    
    # Imposto la strategia di cross-validation.
    # GroupKFold assicura che le lesioni dello stesso paziente non vengano mai divise tra training set e test set
    cv = GroupKFold(n_splits=5)
    
    # Lista vuota per collezionare i punteggi di performance di ogni fold.
    scores = []



    # Itero manualmente attraverso le 5 fold definite da GroupKFold.
    # 'enumerate' tiene traccia del numero della fold corrente.
    for fold, (train_index, test_index) in enumerate(cv.split(features, target, groups)):
        
        # Suddivide i dati in set di training e di test per la fold corrente.
        X_train, X_test = features.iloc[train_index], features.iloc[test_index]
        y_train, y_test = target.iloc[train_index], target.iloc[test_index]

        # Verifica se nel set di test, per ogni target, sono presenti entrambe le classi (0 e 1).
        is_fold_valid = all(y_test[col].nunique() >= 2 for col in y_test.columns)
        
        if not is_fold_valid:
            # Se una fold contiene solo classi positive per un target,
            # il calcolo dell'F1-score fallirebbe. Quindi assegno uno score di 0 (il peggiore)
            # e saltiamo al prossimo ciclo per evitare errori.
            print(f"ATTENZIONE: Fold {fold} del file {csv_name} è invalida e viene assegnato score 0.")
            scores.append(0.0)
            continue

        # Qui presumo che la fold sia valida, quindi inizio il treining
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)
        
        # Calcola l'F1-score
        score = f1_score(y_test, y_pred, average='micro', zero_division=0)
        scores.append(score)

    # Converto la lista di punteggi in un array numpy per facilitare i calcoli.
    scores = np.array(scores)


    return {
        'mean_score': scores.mean(),      # Performance media sulle 5 fold
        'std_score': scores.std(),        # Variabilità della performance
        'scores_per_fold': scores         # Lista dei 5 punteggi individuali
    }


# Lettura dei file

In [3]:
results = {}
print("="*50 + "\nRandom Forest\n" + "="*50)
for name, file_path in datasets.items():
    results[name] = training(file_path, name)

# Stampo i risultati 
for name, metrics in results.items():
    # Estraggo i 5 punteggi per il modello corrente
    scores_per_fold = metrics['scores_per_fold']
    
    # Formatto i punteggi in una stringa pulita
    formatted_scores = [f'{s:.3f}' for s in scores_per_fold]
    
    # Stampo la riga per il modello corrente
    print(f"\nNome CSV: {name}")
    print(f"    scores per forld: {formatted_scores}")
    print(f"    Media e Dev. Std.: {metrics['mean_score']:.3f} ± {metrics['std_score']:.3f}")


Random Forest

Nome CSV: t2_medsam
    scores per forld: ['0.815', '0.720', '0.756', '0.833', '0.619']
    Media e Dev. Std.: 0.749 ± 0.076

Nome CSV: t2_preprocessed
    scores per forld: ['0.792', '0.750', '0.792', '0.744', '0.636']
    Media e Dev. Std.: 0.743 ± 0.057

Nome CSV: t2_original
    scores per forld: ['0.808', '0.792', '0.809', '0.714', '0.651']
    Media e Dev. Std.: 0.755 ± 0.062

Nome CSV: medsam_dynamic
    scores per forld: ['0.792', '0.750', '0.735', '0.720', '0.585']
    Media e Dev. Std.: 0.717 ± 0.070

Nome CSV: preprocessed_dynamic
    scores per forld: ['0.815', '0.708', '0.723', '0.846', '0.564']
    Media e Dev. Std.: 0.731 ± 0.099

Nome CSV: original_dynamic
    scores per forld: ['0.764', '0.735', '0.682', '0.816', '0.619']
    Media e Dev. Std.: 0.723 ± 0.068
